In [ ]:
import glob
import re
import numpy as np

from matplotlib import pyplot as plt

## Display Case 2 Results

This notebook generates Figures 4, 5, 6, and 7 for Case 2 (three-block model).

Before running this notebook, please execute the following script:

```
./run.sh
```


In [ ]:
plt.rcParams['font.size'] = 14
mmin = 0
mmax = 2.5
cwmin = -0.1
cwmax = 0.3

In [ ]:
def lcurve_curvature(lambda_vals, misfit, regularization):
    '''
    Compute L-curve curvature in log–log space.

    Parameters
    ----------
    lambda_vals : array_like
        log10(λ) values.
    misfit : array_like
        log10 of data misfit.
    regularization : array_like
        log10 of regularization term.

    Returns
    -------
    curvature : ndarray
        Curvature of the L-curve. The optimal λ is typically at
        the maximum curvature.

    Notes
    -----
    Curvature is computed from a parametric curve using cubic splines.
    '''
    t = lambda_vals
    x = misfit
    y = regularization

    from scipy.interpolate import CubicSpline

    sx = CubicSpline(t, x)
    sy = CubicSpline(t, y)

    x1 = sx(t, 1)
    x2 = sx(t, 2)
    y1 = sy(t, 1)
    y2 = sy(t, 2)

    denom = (x1**2 + y1**2)**1.5
    denom = np.where(denom == 0, 1e-20, denom)

    curvature = np.abs(x1 * y2 - y1 * x2) / denom

    return curvature

def cross_section_index(direction, cross, x, y, z):
    '''
    Return indices of grid points corresponding to a specified cross-section.

    Parameters
    ----------
    direction : {'ew', 'ns', 'h'}
        Direction of the cross-section:
        'ew' : x–z section (fixed y),
        'ns' : y–z section (fixed x),
        'h'  : horizontal slice (fixed z).
    cross : float
        Target coordinate value. The nearest grid value is used.
    x, y, z : array_like
        Coordinates of the model grid.

    Returns
    -------
    idx : ndarray
        Indices of points belonging to the selected cross-section.

    Notes
    -----
    The cross-section is extracted at the grid location closest to
    the specified coordinate to ensure robustness against floating-point
    mismatch and non-uniform grids.
    '''
    direction = direction.lower()

    x0 = np.unique(x)
    y0 = np.unique(y)
    z0 = np.unique(z)

    def grid_spacing(arr, name):
        if len(arr) < 2:
            raise ValueError(f"{name} grid has insufficient points")
        return np.min(np.diff(arr))

    if direction == 'ew':
        # x(EW)-z cross section
        dy = grid_spacing(y0, "y")
        idx = np.where(np.abs(y - cross) < dy / 2.0)[0]

    elif direction == 'ns':
        # y(NS)-z cross section
        dx = grid_spacing(x0, "x")
        idx = np.where(np.abs(x - cross) < dx / 2.0)[0]

    elif direction == 'h':
        # horizontal slice
        dz = grid_spacing(z0, "z")
        idx = np.where(np.abs(z - cross) < dz / 2.0)[0]

    else:
        raise ValueError("direction must be one of {'ew', 'ns', 'h'}")

    if idx.size == 0:
        raise ValueError("No points found within the specified cross-section range")

    return idx

## Input anomaly (Figure 2)

In [ ]:
### Figure 2 ###

data = np.loadtxt('input_mag.in', delimiter='\t')

x = data[:, 0]
y = data[:, 1]
z = data[:, 2]
f = data[:, 3]

x_unique = np.unique(x)
y_unique = np.unique(y)

nx = len(x_unique)
ny = len(y_unique)

# --- sort ---
idx = np.lexsort((x, y))
x_sorted = x[idx]
y_sorted = y[idx]
f_sorted = f[idx]

# --- reshape ---
X = x_sorted.reshape(ny, nx)
Y = y_sorted.reshape(ny, nx)
F = f_sorted.reshape(ny, nx)

# --- plot ---
fig, ax = plt.subplots(figsize=(8, 8))

pc = ax.pcolormesh(X, Y, F, cmap='jet', shading='auto')
ax.contour(X, Y, F, levels=20, colors='k', linewidths=0.5)

ax.set_aspect('equal')
ax.set_xlabel('x (km)')
ax.set_ylabel('y (km)')

cbar = plt.colorbar(pc, orientation='horizontal', shrink=0.5, pad=0.1)
cbar.set_label('(nT)')

plt.show()

## L-curve and its curvature (Figure 5)

In [ ]:
# --- data ---
w = np.loadtxt('w.vec', skiprows=2)
alpha = 0.9

fn_list = sorted(glob.glob('model_L1L2_*.data'))

nrm_list = []
rss_list = []
lambda_list = []

for fn in fn_list:

    m = re.search(r'model_L1L2_(\d+)\.data', fn)
    if m is None:
        continue
    idx = m.group(1)

    # --- model ---
    res = np.loadtxt(fn)
    b = res[:, 3]
    beta = b * w

    # --- regularization ---
    l2 = 0.5 * np.linalg.norm(beta)**2
    l1 = np.sum(np.abs(beta))
    nrmi = (1 - alpha) * l2 + alpha * l1

    # --- data misfit ---
    res = np.loadtxt(f'recovered_L1L2_{idx}.data')
    r = res[:, 3]
    rssi = 0.5 * np.linalg.norm(f - r)**2

    nrm_list.append(nrmi)
    rss_list.append(rssi)

nrm1 = np.array(nrm_list)
rss1 = np.array(rss_list)

In [ ]:
# L-curve for L1TV inversion
fn_list = sorted(glob.glob('model_L1TV_*.data'))

nrm2 = np.empty(0)
rss2 = np.empty(0)
for fn in fn_list:

    idx = re.split('[_.]', fn)[2]
    if np.char.isdigit(idx) == False:
        continue    

    d = np.loadtxt('regularization_L1TV_{:s}.vec'.format(idx), skiprows=2)
    nrmi = np.abs(d).sum()

    res = np.loadtxt('recovered_L1TV_{:s}.data'.format(idx))
    r = res[:, 3]
    rssi = np.linalg.norm(f - r) / 2.

    nrm2 = np.append(nrm2, nrmi)
    rss2 = np.append(rss2, rssi)

In [ ]:
# --- weights ---
w0 = np.loadtxt('weight_d0.vec', skiprows=2)
wx = np.loadtxt('weight_dx.vec', skiprows=2)
wy = np.loadtxt('weight_dy.vec', skiprows=2)
wz = np.loadtxt('weight_dz.vec', skiprows=2)

ws = np.concatenate([w0, wx, wy, wz])

# --- data ---
fn_list = sorted(glob.glob('model_AdaL1TV_*.data'))

nrm_list = []
rss_list = []

for fn in fn_list:

    m = re.search(r'model_AdaL1TV_(\d+)\.data', fn)
    if m is None:
        continue
    idx = m.group(1)

    # --- regularization ---
    d = np.loadtxt(f'regularization_AdaL1TV_{idx}.vec', skiprows=2)

    if len(d) != len(ws):
        raise ValueError("weight and regularization size mismatch")

    nrmi = np.sum(ws * np.abs(d))

    # --- misfit ---
    res = np.loadtxt(f'recovered_AdaL1TV_{idx}.data')
    r = res[:, 3]
    rssi = 0.5 * np.linalg.norm(f - r)**2

    nrm_list.append(nrmi)
    rss_list.append(rssi)

nrm3 = np.array(nrm_list)
rss3 = np.array(rss_list)

In [ ]:
### Figure 5 (a), (b), (c) ###

fig = plt.figure(figsize=(12, 6))

# ======================
# L1-L2
# ======================
res = np.loadtxt('lambdas_L1L2.data')
lambda_vals = res[:, 1]

lrss1 = np.log10(rss1)
lnrm1 = np.log10(nrm1)

c = lcurve_curvature(lambda_vals, lrss1, lnrm1)
opt = np.argmax(c)

print(f'L1L2: optimal lambda: lambda[{opt}] = {10**lambda_vals[opt]:.2e}')

ax = fig.add_subplot(1, 3, 2)
ax.plot(lnrm1, lrss1, '-o')
ax.plot(lnrm1[opt], lrss1[opt], 'ro')
ax.set_xlabel(r'$(1-\alpha)||\mathbf{\beta}||^2/2+\alpha|\mathbf{\beta}|$')

# ======================
# L1TV
# ======================
res = np.loadtxt('lambdas_L1TV.data')
lambda_vals = res[:, 1]

lrss2 = np.log10(rss2)
lnrm2 = np.log10(nrm2)

c = lcurve_curvature(lambda_vals, lrss2, lnrm2)
opt = np.argmax(c)

print(f'L1TV: optimal lambda: lambda[{opt}] = {10**lambda_vals[opt]:.2e}')

ax = fig.add_subplot(1, 3, 1)
ax.plot(lnrm2, lrss2, '-o')
ax.plot(lnrm2[opt], lrss2[opt], 'ro')
ax.set_ylabel(r'$||\mathbf{f}-\mathbf{X\beta}||^2/2$')
ax.set_xlabel(r'$|\mathbf{\Psi\beta}|$')

# ======================
# AdaL1TV
# ======================
res = np.loadtxt('lambdas_AdaL1TV.data')
lambda_vals = res[:, 1]

lrss3 = np.log10(rss3)
lnrm3 = np.log10(nrm3)

c = lcurve_curvature(lambda_vals, lrss3, lnrm3)
opt = np.argmax(c)

print(f'AdaL1TV: optimal lambda: lambda[{opt}] = {10**lambda_vals[opt]:.2e}')

ax = fig.add_subplot(1, 3, 3)
ax.plot(lnrm3, lrss3, '-o')
ax.plot(lnrm3[opt], lrss3[opt], 'ro')
ax.set_xlabel(r'$\sum w_k|(\mathbf{\Psi\beta})_k|$')

plt.show()

In [ ]:
### Figure 5 (d), (e), (f) ###

fig = plt.figure(figsize=(12, 4))

xticks = np.arange

# L1-L2 curvature
res = np.loadtxt('lambdas_L1L2.data')
lambda_vals = res[:, 1]

lrss1 = np.log10(rss1)
lnrm1 = np.log10(nrm1)

c = lcurve_curvature(lambda_vals, lrss1, lnrm1)
opt = np.where(c == max(c))[0][0]

ax = fig.add_subplot(1, 3, 2)
ax.plot(lambda_vals, c, '-o')
ax.plot(lambda_vals[opt], c[opt], 'ro')
ax.set_xlabel(r'$\log_{10}\, \lambda$')

# L1TV curvature
res = np.loadtxt('lambdas_L1TV.data')
lambda_vals = res[:, 1]

lrss2 = np.log10(rss2)
lnrm2 = np.log10(nrm2)

c = lcurve_curvature(lambda_vals, lrss2, lnrm2)
opt = np.where(c == max(c))[0][0]

ax = fig.add_subplot(1, 3, 1)
ax.plot(lambda_vals, c, '-o')
ax.plot(lambda_vals[opt], c[opt], 'ro')
ax.set_xlabel(r'$\log_{10}\, \lambda$')
ax.set_ylabel('curvature')

# AdaL1TV curvature
res = np.loadtxt('lambdas_L1TV.data')
lambda_vals = res[:, 1]

lrss3 = np.log10(rss3)
lnrm3 = np.log10(nrm3)

c = lcurve_curvature(lambda_vals, lrss3, lnrm3)
opt = np.where(c == max(c))[0][0]

ax = fig.add_subplot(1, 3, 3)
ax.plot(lambda_vals, c, '-o')
ax.plot(lambda_vals[opt], c[opt], 'ro')
ax.set_xlabel(r'$\log_{10}\, \lambda$')

plt.show()

### True model (Figure 1)

In [ ]:
res = np.loadtxt('true.model', delimiter='\t')

x = res[:, 0]
y = res[:, 1]
z = res[:, 2]
b0 = res[:, 3]

x_unique = np.unique(x)
y_unique = np.unique(y)
z_unique = np.unique(z)

nx = len(x_unique)
ny = len(y_unique)
nz = len(z_unique)

print(nx, ny, nz)

In [ ]:
### Figure 1 (b), (c) ###

yc0 = 0.0
yc = y_unique[np.argmin(np.abs(y_unique - yc0))]
idx1 = cross_section_index('EW', yc, x, y, z)

# sort
idx_sort = np.lexsort((x[idx1], z[idx1]))

xs = x[idx1][idx_sort]
zs = z[idx1][idx_sort]
bs = b0[idx1][idx_sort]

xx = xs.reshape(nz, nx)
zz = zs.reshape(nz, nx)
bb = bs.reshape(nz, nx)

# --- NS cross section ---
xc0 = 0.0
xc = y_unique[np.argmin(np.abs(x_unique - xc0))]
idx2 = cross_section_index('NS', xc, x, y, z)

# sort（z -> y）
idx_sort = np.lexsort((y[idx2], z[idx2]))

ys = y[idx2][idx_sort]
zs = z[idx2][idx_sort]
bs = b0[idx2][idx_sort]

yy = ys.reshape(nz, ny)
zz2 = zs.reshape(nz, ny)
bb2 = bs.reshape(nz, ny)

# --- plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# EW
ax = axes[0]
pc = ax.pcolormesh(xx, -zz, bb, cmap='jet', shading='auto')
pc.set_clim(mmin, mmax)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(2.5, 0)
ax.set_aspect('equal')
ax.set_xlabel('y (km)')
ax.set_ylabel('z (km)')
plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.2, label='(A/m)')

# NS
ax = axes[1]
pc = ax.pcolormesh(yy, -zz2, bb2, cmap='jet', shading='auto')
pc.set_clim(mmin, mmax)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(2.5, 0)
ax.set_aspect('equal')
ax.set_xlabel('x (km)')
ax.set_ylabel('z (km)')
plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.2, label='(A/m)')

plt.show()

## Figure 6 and 7

In [ ]:
### Figure 6 (a), (b) ###

res = np.loadtxt('model_L1TV_opt.data', delimiter='\t')
x = res[:, 0]
y = res[:, 1]
z = res[:, 2]
b = res[:, 3]

yc0 = 0.0
yc = y_unique[np.argmin(np.abs(y_unique - yc0))]
idx1 = cross_section_index('EW', yc, x, y, z)

# sort（z -> x）
idx_sort = np.lexsort((x[idx1], z[idx1]))

xs = x[idx1][idx_sort]
zs = z[idx1][idx_sort]
bs = b[idx1][idx_sort]

xx = xs.reshape(nz, nx)
zz = zs.reshape(nz, nx)
bb = bs.reshape(nz, nx)

# --- NS cross section ---
xc0 = 0.0
xc = y_unique[np.argmin(np.abs(x_unique - xc0))]
idx2 = cross_section_index('NS', xc, x, y, z)

# sort（z -> y）
idx_sort = np.lexsort((y[idx2], z[idx2]))

ys = y[idx2][idx_sort]
zs = z[idx2][idx_sort]
bs = b[idx2][idx_sort]

yy = ys.reshape(nz, ny)
zz2 = zs.reshape(nz, ny)
bb2 = bs.reshape(nz, ny)

# --- plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# EW
ax = axes[0]
pc = ax.pcolormesh(xx, -zz, bb, cmap='jet', shading='auto')
pc.set_clim(mmin, mmax)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(2.5, 0)
ax.set_aspect('equal')
ax.set_xlabel('y (km)')
ax.set_ylabel('z (km)')
plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.2, label='(A/m)')

# NS
ax = axes[1]
pc = ax.pcolormesh(yy, -zz2, bb2, cmap='jet', shading='auto')
pc.set_clim(mmin, mmax)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(2.5, 0)
ax.set_aspect('equal')
ax.set_xlabel('x (km)')
ax.set_ylabel('z (km)')
plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.2, label='(A/m)')

plt.show()

In [ ]:
### Figure 6 (c), (d) ###

res = np.loadtxt('model_L1L2_opt.data', delimiter='\t')
b = res[:, 3]

yc0 = 0.0
yc = y_unique[np.argmin(np.abs(y_unique - yc0))]
idx1 = cross_section_index('EW', yc, x, y, z)

# sort（z -> x）
idx_sort = np.lexsort((x[idx1], z[idx1]))

xs = x[idx1][idx_sort]
zs = z[idx1][idx_sort]
bs = b[idx1][idx_sort]

xx = xs.reshape(nz, nx)
zz = zs.reshape(nz, nx)
bb = bs.reshape(nz, nx)

# --- NS cross section ---
xc0 = 0.0
xc = y_unique[np.argmin(np.abs(x_unique - xc0))]
idx2 = cross_section_index('NS', xc, x, y, z)

# sort（z -> y）
idx_sort = np.lexsort((y[idx2], z[idx2]))

ys = y[idx2][idx_sort]
zs = z[idx2][idx_sort]
bs = b[idx2][idx_sort]

yy = ys.reshape(nz, ny)
zz2 = zs.reshape(nz, ny)
bb2 = bs.reshape(nz, ny)

# --- plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# EW
ax = axes[0]
pc = ax.pcolormesh(xx, -zz, bb, cmap='jet', shading='auto')
pc.set_clim(mmin, mmax)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(2.5, 0)
ax.set_aspect('equal')
ax.set_xlabel('y (km)')
ax.set_ylabel('z (km)')
plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.2, label='(A/m)')

# NS
ax = axes[1]
pc = ax.pcolormesh(yy, -zz2, bb2, cmap='jet', shading='auto')
pc.set_clim(mmin, mmax)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(2.5, 0)
ax.set_aspect('equal')
ax.set_xlabel('x (km)')
ax.set_ylabel('z (km)')
plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.2, label='(A/m)')

plt.show()

In [ ]:
### Figure 6 (e), (f) and Figure 7 ###

s = ['0', 'x', 'y', 'z']
data_dict = {}

for key in s:
    fname = f'weight_d{key}.vec'
    
    with open(fname, 'r') as fp:
        lines = fp.readlines()[2:]  # skip headers
    
    d = np.array([float(val.strip()) for val in lines])
    data_dict[key] = d

# extract
d0 = data_dict['0']
dx = data_dict['x']
dy = data_dict['y']
dz = data_dict['z']

res = np.loadtxt('model_AdaL1TV_opt.data', delimiter='\t')
b = res[:, 3]

# =========================
# index of cross-section
# =========================
yc = y_unique[np.argmin(np.abs(y_unique - 0.0))]
idx1 = cross_section_index('EW', yc, x, y, z)

xc = x_unique[np.argmin(np.abs(x_unique - 0.0))]
idx2 = cross_section_index('NS', xc, x, y, z)

# =========================
#           EW
# =========================
idx_sort = np.lexsort((x[idx1], z[idx1]))

xs = x[idx1][idx_sort]
zs = z[idx1][idx_sort]

xx = xs.reshape(nz, nx)
zz = zs.reshape(nz, nx)

bb_ew = b[idx1][idx_sort].reshape(nz, nx)

# =========================
#           NS
# =========================
idx_sort = np.lexsort((y[idx2], z[idx2]))

ys = y[idx2][idx_sort]
zs = z[idx2][idx_sort]

yy = ys.reshape(nz, ny)
zz2 = zs.reshape(nz, ny)

bb_ns = b[idx2][idx_sort].reshape(nz, ny)

# =========================
#           model
# =========================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# EW
ax = axes[0]
pc = ax.pcolormesh(xx, -zz, bb_ew, cmap='jet', shading='auto')
pc.set_clim(mmin, mmax)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(2.5, 0)
ax.set_aspect('equal')
ax.set_xlabel('y(km)')
ax.set_ylabel('z(km)')
plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.2, label='(A/m)')

# NS
ax = axes[1]
pc = ax.pcolormesh(yy, -zz2, bb_ns, cmap='jet', shading='auto')
pc.set_clim(mmin, mmax)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(2.5, 0)
ax.set_aspect('equal')
ax.set_xlabel('x(km)')
ax.set_ylabel('z(km)')
plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.2, label='(A/m)')

# =========================
# weight（EW cross-section）
# =========================
def safe_log10(a):
    return np.log10(np.clip(a, 1e-20, None))

bb0 = d0[idx1][idx_sort].reshape(nz, nx)
bb1 = dx[idx1][idx_sort].reshape(nz, nx)
bb2 = dy[idx1][idx_sort].reshape(nz, nx)
bb3 = dz[idx1][idx_sort].reshape(nz, nx)

B = [bb0, bb1, bb2, bb3]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for i, ax in enumerate(axes.flat):
    pc = ax.pcolormesh(xx, -zz, safe_log10(B[i]), cmap='jet', shading='auto')

    if cwmax > 0:
        pc.set_clim(cwmin, cwmax)

    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(2.5, 0)
    ax.set_aspect('equal')
    ax.set_xlabel('y(km)')
    ax.set_ylabel('z(km)')

    plt.colorbar(pc, ax=ax, orientation='horizontal',
                 shrink=0.5, pad=0.2,
                 label=r'$\log_{10}(\mathbf{w})$')

plt.show()